# Deep Learning for image reconstruction from fMRI response

# Description:

The goal of this project is to implement and evaluate a deep learning model that aims at reconstructing a seen image using the corresponding fMRI response. This notebook describes the whole pipeline step by step.

It consists in three stages:
- Stage 1: retrieving the dataset and forming the fMRI/ground-truth pairs of the training and testing sets.
- Stage 2: Implementing and training the model as a combination of an encoder and a decoder.
- Stage 3: Analysing the performance of the previously trained model through different metrics.



As part of the same project, two other notebooks based on this pipeline are provided to:
- Measure the effect of restricting the data to subsets of ROI
- Measure the effect of reducing the size of the training dataset with self-supervised learning

 # Stage 1: Dataset
Generic Object Decoding (GOD)

The Generic Object Decoding dataset was collected and published by Horikawa & Kamitani (2017) at the Kamitani Lab (Kyoto University). It is publicly available on figshare (article ID 7387130). The raw BIDS-formatted data is also available on OpenNeuro (ds001246).

Experimental design:
Five subjects participated in fMRI sessions while viewing images drawn from ImageNet.
The stimulus set consists of 1250 ImageNet images. 1200 images were used as training stimuli (one presentation per image per subject) and 50 images were held out as test stimuli (35 repeated presentations per image per subject).

Voxel responses are expressed as z-scores computed per voxel per run and restricted to the visual cortex (4466 voxels for Subject1)

Anatomical ROI labels are provided for V1, V2, V3, V4, LOC, FFA, PPA, LVC, HVC and VC.

Data are distributed as HDF5 files (.mat v7.3 format), one per subject. Each file contains a dataSet matrix of shape (4471 x 3450) where rows correspond to voxels (4466) and metadata entries (5), and columns correspond to individual trials. Metadata rows encode trial type, run number, stimulus ID, category index and image index.

This project uses data from Subject1 only. The 1200 training trials provide supervised (image, fMRI) pairs for encoder and decoder training. The 50 averaged test fMRI are used as test set and as unlabeled fMRI data for the self-supervised L_DE loss component described in Beliy et al. (2019).

 Stimuli images are not directly provided with the dataset but can be asked with the form: https://forms.gle/ujvA34948Xg49jdn9


1. Folders and setup

In [ ]:
import os
import requests
from tqdm import tqdm
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import random as rand

In [ ]:
project = 'dataset'
fMRI = 'dataset/fMRI'
images = 'dataset/images'

for d in [project, fMRI, images,
          os.path.join(images, 'training'),
          os.path.join(images, 'test'),
          'dataset/unlabled',
          'models/encoders',
          'models/decoders',
          'results/nway_accuracy',
          'results/snr_curves',
          'results/other_metrics',
          'results/reconstructions',
          'results/data_size_experiment',
          'results/roi_experiments']:
    os.makedirs(d, exist_ok=True)

2. Download fMRI data


Subject1.mat is in HDF5 v7.3 format.
 It contains:

- dataSet with shape (4471, 3450) and float64 dtype.
- A metaData group containing:
  -  description dataset with shape (21, 1) and object dtype.
  - key dataset with shape (21, 1) and object dtype.
  - value dataset with shape (4471, 21) and float64 dtype.

Source : figshare via S3 AWS (URL obtained through https://github.com/KamitaniLab/GenericObjectDecoding)

In [ ]:
MAT_URL  = 'https://s3-eu-west-1.amazonaws.com/pfigshare-u-files/13663487/Subject1.mat'
MAT_PATH = os.path.join(fMRI, 'Subject1.mat')

if os.path.exists(MAT_PATH):
    print(f'Subject1 already downloaded')
else:
    print('Downloading Subject1.mat...')
    r = requests.get(
        MAT_URL, stream=True,
        headers={'User-Agent': 'Mozilla/5.0'}
    )
    total = int(r.headers.get('content-length', 0))
    with open(MAT_PATH, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as bar:
        for chunk in r.iter_content(8192):
            f.write(chunk)
            bar.update(len(chunk))
    print(f'\n Subject1 downloaded :{os.path.getsize(MAT_PATH)/1e6:.0f} MB')

3. Extract fMRI data

In [ ]:
with h5py.File(MAT_PATH, 'r') as f:
    data = f['dataSet'][:]
    meta = f['metaData/value'][:]
    key_refs = f['metaData/key'][:].flatten()
    keys = [''.join(chr(c[0]) for c in f[ref][:]) for ref in key_refs]

print(f'dataSet shape : {data.shape}')
print(f'metaData shape : {meta.shape}')
print(f'Columns in metaData : {keys}')


In [ ]:
# data contains lines with voxels and lines with metadata
voxeldata_col = keys.index('VoxelData')
is_voxel = meta[:, voxeldata_col] == 1.0
is_other = ~is_voxel

print(f'True voxels : {is_voxel.sum()}')
print(f'Metadata lines : {is_other.sum()}')

other_indices = np.where(is_other)[0]
for i in other_indices:
  row = data[i, :]
  meta_row = meta[i, :]
  active = [keys[k] for k in range(len(keys)) if meta_row[k] == 1.0]
  unique = np.unique(row[~np.isnan(row)])
  print(f'  line {i}, meta={active}')

In [ ]:
fmri_all = data[is_voxel, :].T.astype(np.float32)  # (3450, 4466)
datatype = data[0, :].astype(int) # 1=train, 2=test
run = data[1, :].astype(int) # run number
image_index = data[4470, :] # index image

#test and train
mask_train = datatype == 1
mask_test = datatype == 2

fmri_train = fmri_all[mask_train]
fmri_test_raw = fmri_all[mask_test]
img_idx_train = image_index[mask_train]
img_idx_test = image_index[mask_test]

# for the test images, each image was seen 35 times
# so we can compute the average fMRI response for each image
unique_test_imgs = np.unique(img_idx_test[~np.isnan(img_idx_test)]).astype(int)
fmri_test = np.array([fmri_test_raw[img_idx_test == img].mean(axis=0) for img in unique_test_imgs])

# Clip outliers in fMRI
CLIP_VAL = 20.0

fmri_train = np.clip(fmri_train, -CLIP_VAL, CLIP_VAL).astype(np.float32)
fmri_test = np.clip(fmri_test, -CLIP_VAL, CLIP_VAL).astype(np.float32)
fmri_test_raw = np.clip(fmri_test_raw, -CLIP_VAL, CLIP_VAL).astype(np.float32)

print('Data :')
print(f'  fmri_train : {fmri_train.shape} ({mask_train.sum()} trials)')
print(f'  fmri_test_raw : {fmri_test_raw.shape} ({mask_test.sum()} trials)')
print(f'  fmri_test : {fmri_test.shape} (50 images averaged on 35 runs)')

4. Download stimulus

Due to copyright reasons on ImageNet, the images used as stimuli are not provided in the dataset, but a zip file contraining the images can be requested with the following form: https://forms.gle/ujvA34948Xg49jdn9

The following cell allows to download and unzip the folder. The file must be placed at the root of the repository.

In [ ]:
import zipfile, getpass

ZIP_PATH = 'images_passwd.zip'  # zip should be placed at the root of the repository
password = getpass.getpass('Zip password : ')

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('dataset', pwd=password.encode('utf-8'))

print('Extracted successfully')

In [ ]:
EXTRACT_DIR = 'dataset'
test_dir = os.path.join(EXTRACT_DIR, 'images', 'test')
train_dir = os.path.join(EXTRACT_DIR, 'images', 'training')
test_id_csv = os.path.join(EXTRACT_DIR, 'images', 'image_test_id.csv')
train_id_csv = os.path.join(EXTRACT_DIR, 'images', 'image_training_id.csv')

ids_train = pd.read_csv(train_id_csv, header=None, names=['img_idx', 'filename'])
ids_test = pd.read_csv(test_id_csv,  header=None, names=['img_idx', 'filename'])

5. Building the fMRI/image pairs

In [ ]:
# image_index in the fMRI = position in the CSV

train_idx_to_path = {}
for i, row in ids_train.iterrows():
    position = i + 1
    filename = str(row['filename']).strip()
    path = os.path.join(train_dir, filename)
    if os.path.exists(path):
        train_idx_to_path[position] = path

test_idx_to_path = {}
for i, row in ids_test.iterrows():
    position = i + 1
    filename = str(row['filename']).strip()
    path = os.path.join(test_dir, filename)
    if os.path.exists(path):
        test_idx_to_path[position] = path

print(f'Train : {len(train_idx_to_path)}/1200')
print(f'Test : {len(test_idx_to_path)}/50')

In [ ]:
def build_pairs(fmri_data, img_indices, idx_to_path):
    pairs = []
    for trial_idx, img_pos in enumerate(img_indices):
        img_pos = int(img_pos)
        if img_pos in idx_to_path:
            pairs.append({'trial_idx': trial_idx, 'img_pos': img_pos,'fmri': fmri_data[trial_idx],'img_path': idx_to_path[img_pos],})
    return pairs

pairs_train = build_pairs(fmri_train, img_idx_train, train_idx_to_path)
pairs_test  = build_pairs(fmri_test,  unique_test_imgs, test_idx_to_path)

print(f'Built pairs :')
print(f'  Train : {len(pairs_train)}')
print(f'  Test : {len(pairs_test)}')

In [ ]:
# Display some fMRI/image pairs
indices = rand.sample(range(len(pairs_train)), 4)
pairs_display = [pairs_train[i] for i in indices]

#training set
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
fig.suptitle('fMRI / image pairs  (training set) \n', fontsize=24)

for c, pair in enumerate(pairs_display):
  #img
  img = Image.open(pair['img_path']).convert('RGB')
  axes[0, c].imshow(img)
  axes[0, c].set_title(f'trial {pair["trial_idx"]}', fontsize=14)
  axes[0, c].axis('off')

  #fMRI
  fmri = pair['fmri']
  n = int(np.sqrt(fmri.shape[0]))
  fmri = fmri[:n*n]
  fmri = fmri.reshape(n, n)
  axes[1, c].imshow(fmri, cmap='RdBu_r', aspect='auto', vmin=-3, vmax=3)
  axes[1, c].set_title(' ', fontsize=14)
  axes[1, c].axis('off')

plt.tight_layout()
plt.show()

print("\n")
#testing set

indices = rand.sample(range(len(pairs_test)), 4)
pairs_display = [pairs_test[i] for i in indices]

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
fig.suptitle('fMRI / image pairs  (testing set)\n', fontsize=24)

for c, pair in enumerate(pairs_display):
  #img
  img = Image.open(pair['img_path']).convert('RGB')
  axes[0, c].imshow(img)
  axes[0, c].set_title(f'trial {pair["trial_idx"]}', fontsize=14)
  axes[0, c].axis('off')

  #fMRI
  fmri = pair['fmri']
  n = int(np.sqrt(fmri.shape[0]))
  fmri = fmri[:n*n]
  fmri = fmri.reshape(n, n)
  axes[1, c].imshow(fmri, cmap='RdBu_r', aspect='auto', vmin=-3, vmax=3)
  axes[1, c].set_title(' ', fontsize=14)
  axes[1, c].axis('off')

plt.tight_layout()
plt.show()

#Stage 2: Encoder-Decoder

The model consists of an encoder and a decoder.

The decoder is the part of the model that is trained to predict an image from the voxels of an fMRI.

To improve the decoder during training, a self-supervised approach is used additionnaly to the classical supervised method. An encoder is trained to predict "new" fMRI responses from images and is then used with the decoder in an image-to-image architecture to increase the number of fMRI data the decoder can be trained on. Furthermore, the encoder can be placed after the decoder to create a fMRI-to-fMRI architecture that can be trained on the fMRI of the test set, without the model ever seeing the corresponding image.

The architectures of the encoder and the decoder are based on the paper :

"From voxels to pixels and back: Self-supervision in
natural-image reconstruction from fMRI" (Beliy et al., 2019)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torchvision.transforms as T
import torchvision.models as models
import urllib.request, zipfile

The training of the models requires a gpu for shorter computing time.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')

1. Creation of the training and testing datasets

Three datasets for training have to be created:
- the supervised dataset: pairs of fMRIs and their corresponding image (used to train both the encoder and the decoder)
- the image dataset for unsupervised training (E-D architecture): images from TinyImageNet
- the fMRI dataset for unsupervised training (D-E architecture): fMRIs from the testing set

In [ ]:
UNLABELED_DIR = 'dataset/unlabled'
os.makedirs(UNLABELED_DIR, exist_ok=True)

TINY_URL  = 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'
TINY_ZIP  = os.path.join(UNLABELED_DIR, 'tiny-imagenet-200.zip')
TINY_DIR  = os.path.join(UNLABELED_DIR, 'tiny-imagenet-200')

if os.path.exists(TINY_DIR) and len(os.listdir(TINY_DIR)) > 0:
    print('TinyImageNet already there')
else:
    print('Downloading TinyImageNet...')
    import requests
    r = requests.get(TINY_URL, stream=True)
    total = int(r.headers.get('content-length', 0))
    with open(TINY_ZIP, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as bar:
        for chunk in r.iter_content(8192):
            f.write(chunk)
            bar.update(len(chunk))
    print('Extraction...')
    with zipfile.ZipFile(TINY_ZIP, 'r') as z:
        z.extractall(UNLABELED_DIR)
    print('Success')

unlabeled_paths = []
for root, dirs, files in os.walk(TINY_DIR):
    for fname in files:
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            unlabeled_paths.append(os.path.join(root, fname))

print(f'Unlabled images available : {len(unlabeled_paths)}')

In [ ]:
IMG_SIZE = 112

transform_train = T.Compose([
    T.Resize((IMG_SIZE + 16, IMG_SIZE + 16)),
    T.RandomCrop((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

transform_test = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

# Denormalisation for visualisation
def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1)


class PairedDataset(Dataset):
    def __init__(self, pairs, transform):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        pair = self.pairs[i]
        img = Image.open(pair['img_path']).convert('RGB')
        return self.transform(img), torch.FloatTensor(pair['fmri'])


class UnlabeledImageDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]).convert('RGB')
            return self.transform(img)
        except Exception:
            return torch.zeros(3, IMG_SIZE, IMG_SIZE)


class FMRIDataset(Dataset):
    def __init__(self, fmri_data):
        self.fmri = fmri_data

    def __len__(self):
        return len(self.fmri)

    def __getitem__(self, i):
        return torch.FloatTensor(self.fmri[i])


dataset_train = PairedDataset(pairs_train, transform_train)
dataset_test_vis = PairedDataset(pairs_test, transform_test)
dataset_unlabeled = UnlabeledImageDataset(unlabeled_paths, transform_train)
dataset_fmri_test = FMRIDataset(fmri_test)

print(f'Dataset train : {len(dataset_train)} pairs')
print(f'Dataset test : {len(dataset_test_vis)} pairs')
print(f'Dataset unlabeled : {len(dataset_unlabeled)} images')
print(f'Dataset fMRI test : {len(dataset_fmri_test)} vectors')

2. Architecture

Described in Beliy et al. (2019)

In [ ]:
N_VOXELS = 4466

class Encoder(nn.Module):
    """
    Encoder E : (3x112x112) to (n_voxels,)
    Architecture (Beliy et al.) :
    - Feature extraction: AlexNet conv1 (frozen)
    - 2 blocs conv3x3 stride=2, 32 chanels, BN, ReLU
    - FC to n_voxels
    """
    def __init__(self, n_voxels):
        super().__init__()

        alexnet = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
        self.alexnet_conv1 = alexnet.features[0]
        for p in self.alexnet_conv1.parameters():
            p.requires_grad = False

        self.bn_input = nn.BatchNorm2d(64)

        self.conv_blocks = nn.Sequential(
            nn.Conv2d(64, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU()
        )

        # Compute dimension after convolutions
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 112, 112)
            dummy = self.alexnet_conv1(dummy)
            dummy = self.conv_blocks(dummy)
            flat_dim = dummy.view(1, -1).shape[1]

        self.fc = nn.Linear(flat_dim, n_voxels)

    def forward(self, x):
      with torch.no_grad():
        x = self.alexnet_conv1(x)
      x = self.bn_input(x)
      x = self.conv_blocks(x)
      x = x.view(x.size(0), -1)
      return self.fc(x)


class Decoder(nn.Module):
    """
    Decoder D : fMRI (n_voxels,) to image (3x112x112)

    Architecture (Beliy et al.) :
    - FC reshape to (64, 14, 14)
    - 3 blocs conv3x3, stride=1, 64 channels, ReLU, Upsample x2, BN
    - conv3x3 + Sigmoid : (3, 112, 112)
    """
    def __init__(self, n_voxels):
        super().__init__()

        self.fc = nn.Linear(n_voxels, 64 * 14 * 14)

        self.decoder_blocks = nn.Sequential(
            # Bloc 1 : 14x14 to 28x28
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.BatchNorm2d(64),
            # Bloc 2 : 28x28 to 56x56
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.BatchNorm2d(64),
            # Bloc 3 : 56x56 to 112x112
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.BatchNorm2d(64),
        )

        self.output_conv = nn.Sequential(
            nn.Conv2d(64, 3, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, fmri):
        x = self.fc(fmri)
        x = x.view(-1, 64, 14, 14)
        x = self.decoder_blocks(x)
        return self.output_conv(x)


encoder = Encoder(N_VOXELS).to(DEVICE)
decoder = Decoder(N_VOXELS).to(DEVICE)

enc_params = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
dec_params = sum(p.numel() for p in decoder.parameters() if p.requires_grad)
print(f'Encodeur weights to train: {enc_params:,}')
print(f'Decodeur weights to train: {dec_params:,}')

3. Loss functions

In [ ]:
alpha = 0.9

def fmri_loss(pred, target):
    mse = F.mse_loss(pred, target)
    cos = F.cosine_similarity(pred, target, dim=1).mean()
    return alpha * mse - (1 - alpha) * cos

class VGGFeatures(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features
        self.slice1 = vgg[:2]
        self.slice2 = vgg[:7]
        for p in self.parameters():
            p.requires_grad = False

    def forward(self, x):
        f1 = self.slice1(x)
        f2 = self.slice2(x)
        return f1, f2

vgg_features = VGGFeatures().to(DEVICE)

def total_variation(img):
    return (torch.abs(img[:, :, 1:, :] - img[:, :, :-1, :]).mean() +
            torch.abs(img[:, :, :, 1:] - img[:, :, :, :-1]).mean())

def image_loss(recon, target):
    l_rgb  = F.l1_loss(recon, target)
    f1, f2 = vgg_features(recon)
    g1, g2 = vgg_features(target)
    l_feat = F.mse_loss(f1, g1) + F.mse_loss(f2, g2)
    l_tv = total_variation(recon)

    l_rgb_coef = 1
    l_feat_coef = 0.1
    l_tv_coef = 0.001
    return l_rgb_coef * l_rgb + l_feat_coef * l_feat + l_tv_coef * l_tv

4. Training of the encoder

In [ ]:
BATCH_SIZE_ENC = 32
N_EPOCHS_ENC = 80

loader_train_enc = DataLoader(dataset_train, batch_size=BATCH_SIZE_ENC, shuffle=True, num_workers=2, pin_memory=True)
optimizer_enc = torch.optim.SGD(encoder.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
scheduler_enc = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_enc, T_max=N_EPOCHS_ENC + 20)

enc_losses = []
for epoch in range(N_EPOCHS_ENC):
    encoder.train()
    epoch_loss = 0
    for i, (img, fmri) in enumerate(loader_train_enc):
        img, fmri = img.to(DEVICE), fmri.to(DEVICE)
        optimizer_enc.zero_grad()
        fmri_pred = encoder(img)
        loss = fmri_loss(fmri_pred, fmri)
        loss.backward()
        optimizer_enc.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(loader_train_enc)
    scheduler_enc.step()
    enc_losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        encoder.eval()
        with torch.no_grad():
            imgs_sample, fmris_sample = next(iter(DataLoader(dataset_train, batch_size=64, shuffle=False)))
            fmri_pred = encoder(imgs_sample.to(DEVICE)).cpu()
            cos_sim = F.cosine_similarity(fmri_pred, fmris_sample, dim=1).mean().item()
        encoder.train()
        print(f'  Epoch {epoch+1:3d}/{N_EPOCHS_ENC}, '
              f'Loss : {avg_loss:.4f}, '
              f'Cos sim : {cos_sim:.4f}, '
              f'LR : {scheduler_enc.get_last_lr()[0]:.6f}')


5. Training of the decoder

In [ ]:
#freeze the encoder
encoder.eval()
for p in encoder.parameters():
  p.requires_grad = False

BATCH_SUPERVISED = 36 # 60%
BATCH_UNLABELED = 12 # 20%
BATCH_FMRI = 12 # 20%
N_EPOCHS_DEC = 150

loader_supervised = DataLoader(dataset_train, batch_size=BATCH_SUPERVISED, shuffle=True, num_workers=2, pin_memory=True)
loader_unlabeled = DataLoader(dataset_unlabeled, batch_size=BATCH_UNLABELED, shuffle=True, num_workers=2, pin_memory=True)
loader_fmri = DataLoader(dataset_fmri_test, batch_size=BATCH_FMRI, shuffle=True, num_workers=2, pin_memory=True)

optimizer_dec = torch.optim.Adam(decoder.parameters(), lr=0.001, weight_decay=1e-4)
scheduler_dec = torch.optim.lr_scheduler.StepLR(optimizer_dec, step_size=30, gamma=0.2)




Train the decoder with 3 losses:
- l_d : decoder loss (supervised) from fMRI to image
- l_ed : encoder-decoder loss (self-supervised) from image to image
- l_de : decoder-encoder loss (self-supervised) from fMRI to fMRI

lambda_d, lambda_ed and lambda_de can be adjusted to compare supervised vs self-supervised methods.

In [ ]:
dec_losses = {'total': [], 'L_D': [], 'L_ED': [], 'L_DE': []}

lambda_d = 1
lambda_ed = 1
lambda_de = 0

def infinite_loader(loader):
    while True:
        for batch in loader:
            yield batch

iter_unlabeled = infinite_loader(loader_unlabeled)
iter_fmri_test = infinite_loader(loader_fmri)

for epoch in range(N_EPOCHS_DEC):
    decoder.train()
    epoch_total = 0
    epoch_ld = 0
    epoch_led = 0
    epoch_lde = 0
    n_steps = 0

    for imgs_sup, fmris_sup in loader_supervised:
        imgs_sup  = imgs_sup.to(DEVICE)
        fmris_sup = fmris_sup.to(DEVICE)
        imgs_unlab = next(iter_unlabeled).to(DEVICE)
        fmris_test_batch = next(iter_fmri_test).to(DEVICE)

        optimizer_dec.zero_grad()

        recon_sup = decoder(fmris_sup)
        imgs_sup_01 = (imgs_sup * torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(DEVICE) + torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(DEVICE)).clamp(0, 1)
        l_d = image_loss(recon_sup, imgs_sup_01)

        with torch.no_grad():
            fmri_pred_unlab = encoder(imgs_unlab)
        recon_unlab = decoder(fmri_pred_unlab)
        imgs_unlab_01 = (imgs_unlab * torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(DEVICE) + torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(DEVICE)).clamp(0, 1)
        l_ed = image_loss(recon_unlab, imgs_unlab_01)

        recon_from_fmri = decoder(fmris_test_batch)
        fmri_reencoded = encoder(recon_from_fmri)
        l_de = fmri_loss(fmri_reencoded, fmris_test_batch)

        total_loss = lambda_d * l_d + lambda_ed * l_ed + lambda_de * l_de
        total_loss.backward()

        nn.utils.clip_grad_norm_(decoder.parameters(), max_norm=1.0)
        optimizer_dec.step()

        epoch_total += total_loss.item()
        epoch_ld += l_d.item()
        epoch_led += l_ed.item()
        epoch_lde += l_de.item()
        n_steps += 1

    scheduler_dec.step()
    avg_total = epoch_total / n_steps
    dec_losses['total'].append(avg_total)
    dec_losses['L_D'].append(epoch_ld / n_steps)
    dec_losses['L_ED'].append(epoch_led / n_steps)
    dec_losses['L_DE'].append(epoch_lde / n_steps)

    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:3d}/{N_EPOCHS_DEC} | Total={avg_total:.4f} '
              f'L_D={dec_losses["L_D"][-1]:.4f} '
              f'L_ED={dec_losses["L_ED"][-1]:.4f} '
              f'L_DE={dec_losses["L_DE"][-1]:.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Decoder losses', fontsize=12)

axes[0].plot(dec_losses['total'], color='black', lw=1.5, label='Total')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Total Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(dec_losses['L_D'],  color='steelblue',  lw=1.5, label='L_D (supervised)')
axes[1].plot(dec_losses['L_ED'], color='darkorange',  lw=1.5, label='L_ED (unlabeled images)')
axes[1].plot(dec_losses['L_DE'], color='forestgreen', lw=1.5, label='L_DE (fMRI test)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('loss components')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

6. Save the models and display the results on the test set

This cell is to save the models for later use. Choose the name of the encoder and the decoder. Both will be saved in different folders: models/encoders and models/decoders

In [ ]:
#choose the name
model_name = 'model_image_self_supervised'
encoder_name = f'{model_name}_encoder.pt'
decoder_name = f'{model_name}_decoder.pt'

In [ ]:
results_dir = 'results'
encoder_dir = 'models/encoders'
decoder_dir = 'models/decoders'
for d in [results_dir, encoder_dir, decoder_dir,
          os.path.join(results_dir, 'reconstructions')]:
    os.makedirs(d, exist_ok=True)

encoder_path = os.path.join(encoder_dir, encoder_name)
decoder_path = os.path.join(decoder_dir, decoder_name)
torch.save(encoder.state_dict(), encoder_path)
torch.save(decoder.state_dict(), decoder_path)
print(f'Encoder saved : {encoder_path}')
print(f'Decoder saved : {decoder_path}')

fig_enc, ax_enc = plt.subplots(figsize=(8, 4))
ax_enc.plot(enc_losses, color='steelblue', lw=1.5)
ax_enc.set_xlabel('Epoch')
ax_enc.set_ylabel('Loss')
ax_enc.set_title(f'Encoder loss : {model_name}')
ax_enc.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig_dec, axes_dec = plt.subplots(1, 2, figsize=(14, 4))
fig_dec.suptitle(f'Decoder losses : {model_name}', fontsize=12)
axes_dec[0].plot(dec_losses['total'], color='black', lw=1.5)
axes_dec[0].set_xlabel('Epoch')
axes_dec[0].set_ylabel('Loss')
axes_dec[0].set_title('Total Loss')
axes_dec[0].grid(True, alpha=0.3)
axes_dec[1].plot(dec_losses['L_D'], color='steelblue', lw=1.5, label='L_D')
axes_dec[1].plot(dec_losses['L_ED'], color='darkorange', lw=1.5, label='L_ED')
axes_dec[1].plot(dec_losses['L_DE'], color='forestgreen', lw=1.5, label='L_DE')
axes_dec[1].set_xlabel('Epoch')
axes_dec[1].set_ylabel('Loss')
axes_dec[1].set_title('Loss components')
axes_dec[1].legend(fontsize=8)
axes_dec[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

encoder.eval()
decoder.eval()

loader_full = DataLoader(dataset_test_vis, batch_size=50, shuffle=False)
imgs_gt, fmris_gt = next(iter(loader_full))

with torch.no_grad():
    recons = decoder(fmris_gt.to(DEVICE)).cpu()

n_cols = 10
n_rows = (50 // n_cols) * 2

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2, n_rows * 2))
fig.suptitle(f'All test reconstructions : {model_name}', fontsize=12, y=1.01)

for i in range(50):
    col = i % n_cols
    row_orig = (i // n_cols) * 2
    row_recon = row_orig + 1

    axes[row_orig, col].imshow(denormalize(imgs_gt[i]).permute(1, 2, 0))
    axes[row_orig, col].axis('off')
    axes[row_orig, col].set_title(f'#{i+1}', fontsize=7)

    axes[row_recon, col].imshow(recons[i].permute(1, 2, 0).clamp(0, 1))
    axes[row_recon, col].axis('off')

for i in range(0, n_rows, 2):
    axes[i, 0].set_ylabel('Original', fontsize=8, rotation=90, labelpad=4)
    axes[i + 1, 0].set_ylabel('Reconstruction', fontsize=8, rotation=90, labelpad=4)

plt.tight_layout()
recon_all_path = os.path.join(results_dir, 'reconstructions', f'{decoder_name[:-3]}_all50.png')
fig.savefig(recon_all_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved : {recon_all_path}')
print(f'Loss curves saved in : {os.path.join(results_dir, "loss_curves")}')


#Stage 3: Analysis

In [ ]:
LOAD_EXISTING_MODEL = False

if LOAD_EXISTING_MODEL:
    encoder_to_load = 'model_name_encoder.pt'
    decoder_to_load = 'model_name_decoder.pt'
    encoder_load_path = os.path.join('models/encoders', encoder_to_load)
    decoder_load_path = os.path.join('models/decoders', decoder_to_load)
    encoder = Encoder(N_VOXELS).to(DEVICE)
    decoder = Decoder(N_VOXELS).to(DEVICE)
    encoder.load_state_dict(torch.load(encoder_load_path, map_location=DEVICE))
    decoder.load_state_dict(torch.load(decoder_load_path, map_location=DEVICE))
    model_name = encoder_to_load.split('_ep')[0]
    encoder_name = encoder_to_load
    decoder_name = decoder_to_load
    print(f'Loaded encoder : {encoder_load_path}')
    print(f'Loaded decoder : {decoder_load_path}')
else:
    print(f'Using freshly trained models : {model_name}')

encoder.eval()
decoder.eval()
print(f'Active model : {model_name}')

results_dir = 'results'
for d in [os.path.join(results_dir, 'nway_accuracy'),
          os.path.join(results_dir, 'snr_curves'),
          os.path.join(results_dir, 'other_metrics')]:
    os.makedirs(d, exist_ok=True)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim_matrix
from scipy.stats import pearsonr
import itertools, random
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
import numpy as np
from transformers import CLIPModel, CLIPProcessor

1. Evaluate the N-way accuracy of the model

In [ ]:
def nway_identification(decoder, dataset_test, n_way=2, n_trials=1000, device='cuda'):
    decoder.eval()
    all_imgs, all_fmris = [], []
    for imgs, fmris in DataLoader(dataset_test, batch_size=50, shuffle=False):
        all_imgs.append(imgs)
        all_fmris.append(fmris)
    all_imgs = torch.cat(all_imgs)
    all_fmris = torch.cat(all_fmris)
    n_test = len(all_imgs)
    correct = 0
    with torch.no_grad():
        recons = decoder(all_fmris.to(device)).cpu()
    for k in range(n_trials):
        target_idx = random.randint(0, n_test - 1)
        recon = recons[target_idx].numpy().flatten()
        distractors = random.sample([i for i in range(n_test) if i != target_idx], n_way - 1)
        candidates = [target_idx] + distractors
        random.shuffle(candidates)
        corrs = []
        for c in candidates:
            orig = all_imgs[c].numpy().flatten()
            r, _ = pearsonr(recon, orig)
            corrs.append(r)
        if candidates[np.argmax(corrs)] == target_idx:
            correct += 1
    return correct / n_trials


accs = []
for n in [2, 5, 10]:
    acc = nway_identification(decoder, dataset_test_vis, n_way=n, n_trials=1000, device=DEVICE)
    accs.append(acc)
    print(f'{n}-way identification accuracy : {acc*100:.1f}%  (chance = {100/n:.1f}%)')


In [ ]:
acc_Beliy = [85.3, 68, 56]
chance = [50, 20, 10]
acc_model = [a * 100 for a in accs]
labels = ['2-way', '5-way', '10-way']
x = np.arange(len(labels))
width = 0.25

fig_nway, ax_nway = plt.subplots(figsize=(6, 4))
ax_nway.bar(x - width, acc_model, width, label='Our model')
ax_nway.bar(x, acc_Beliy, width, label='Beliy et al.')
ax_nway.bar(x + width, chance, width, label='Chance')
ax_nway.set_ylabel('Accuracy (%)')
ax_nway.set_xlabel('Identification task')
ax_nway.set_title(f'N-way accuracy :{model_name}')
ax_nway.set_xticks(x)
ax_nway.set_xticklabels(labels)
ax_nway.set_ylim(0, 100)
ax_nway.legend()
plt.tight_layout()
nway_path = os.path.join(results_dir, 'nway_accuracy', f'{decoder_name[:-3]}_nway.png')
fig_nway.savefig(nway_path, dpi=120)
plt.show()
print(f'Saved : {nway_path}')


2.  Analyze the impact of averaging the test fMRI of the 35 iterations.

In [ ]:
print('Accuracy depending on the number of fMRIs to make the average :')

n_reps_list = [1, 5, 10, 20, 35]
acc_2way_list = []
acc_10way_list = []

for n_reps in n_reps_list:

    fmri_avg_nreps = np.array([fmri_test_raw[img_idx_test == img][:n_reps].mean(axis=0) for img in np.unique(img_idx_test[~np.isnan(img_idx_test)]).astype(int)])
    fmri_avg_nreps = np.clip(fmri_avg_nreps, -20.0, 20.0).astype(np.float32)

    pairs_nreps = []
    for i, img_pos in enumerate(np.unique(img_idx_test[~np.isnan(img_idx_test)]).astype(int)):
        if img_pos in test_idx_to_path:
            pairs_nreps.append({
                'fmri' : fmri_avg_nreps[i],
                'img_path' : test_idx_to_path[img_pos],
            })
    dataset_nreps = PairedDataset(pairs_nreps, transform_test)

    acc_2way = nway_identification(decoder, dataset_nreps, n_way=2, n_trials=1000, device=DEVICE)
    acc_10way = nway_identification(decoder, dataset_nreps, n_way=10, n_trials=1000, device=DEVICE)
    acc_2way_list.append(acc_2way)
    acc_10way_list.append(acc_10way)
    print(f'  {n_reps:2d} reps, 2-way: {acc_2way*100:.1f}%, 10-way: {acc_10way*100:.1f}%')

In [ ]:
fig_snr, ax_snr = plt.subplots(figsize=(6, 4))

ax_snr.plot(n_reps_list, np.array(acc_2way_list)*100, 'o-', color='steelblue',
            linewidth=2, markersize=7, label='2-way identification')
ax_snr.plot(n_reps_list, np.array(acc_10way_list)*100, 's-', color='darkorange',
            linewidth=2, markersize=7, label='10-way identification')
ax_snr.axhline(50, color='steelblue', linestyle='--', alpha=0.4, label='Chance 2-way (50%)')
ax_snr.axhline(10, color='darkorange', linestyle='--', alpha=0.4, label='Chance 10-way (10%)')
ax_snr.set_xlabel('Number of repetitions', fontsize=10)
ax_snr.set_ylabel('Identification accuracy (%)', fontsize=10)
ax_snr.set_title(f'Impact of repetitions : {model_name}', fontsize=10)
ax_snr.set_xticks(n_reps_list)
ax_snr.legend(fontsize=7, loc='lower right')
ax_snr.set_ylim(0, 100)
ax_snr.grid(True, alpha=0.3)
plt.tight_layout()
snr_path = os.path.join(results_dir, 'snr_curves', f'{decoder_name[:-3]}_snr.png')
fig_snr.savefig(snr_path, dpi=120)
plt.show()
print(f'Saved : {snr_path}')


3. Other metrics to measure reconstruction quality

Pixel correlation: pearson correlation between pixels of the original image and the reconstruction

SSIM (structural similarity index): measures the similarity in the structure, rather than comparing pixel by pixel

Cosine similarity CLIP: measures semantic similarity in CLIP space. Focuses on high level features.

In [ ]:
encoder.eval()
decoder.eval()

loader_full = DataLoader(dataset_test_vis, batch_size=50, shuffle=False)
imgs_gt, fmris_gt = next(iter(loader_full))

with torch.no_grad():
    recons = decoder(fmris_gt.to(DEVICE)).cpu()

imgs_01 = denormalize(imgs_gt)

In [ ]:
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE)
clip_proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
clip_model.eval()

In [ ]:
def compute_pixcorr(recons, imgs_01):
    scores = []
    for i in range(len(recons)):
        orig = imgs_01[i].numpy().flatten()
        recon = recons[i].numpy().flatten()
        r, _ = pearsonr(orig, recon)
        scores.append(r)
    return np.mean(scores)

def compute_ssim(recons, imgs_01):
    scores = []
    for i in range(len(recons)):
        orig = imgs_01[i].permute(1,2,0).numpy()
        recon = recons[i].permute(1,2,0).numpy()
        scores.append(ssim(orig, recon, channel_axis=2, data_range=1.0))
    return np.mean(scores)

def compute_clip_similarity(recons, imgs_01, clip_model, clip_proc, device):
    scores = []
    for i in range(len(recons)):
        orig_pil = Image.fromarray((imgs_01[i].permute(1,2,0).numpy()*255).astype(np.uint8))
        recon_pil = Image.fromarray((recons[i].permute(1,2,0).clamp(0,1).numpy()*255).astype(np.uint8))
        with torch.no_grad():
            f_orig = clip_model.vision_model(**clip_proc(images=orig_pil,  return_tensors='pt').to(device)).pooler_output
            f_recon = clip_model.vision_model(**clip_proc(images=recon_pil, return_tensors='pt').to(device)).pooler_output
            f_orig = clip_model.visual_projection(f_orig)
            f_recon = clip_model.visual_projection(f_recon)
            f_orig = f_orig  / f_orig.norm(dim=-1, keepdim=True)
            f_recon = f_recon / f_recon.norm(dim=-1, keepdim=True)
            scores.append(F.cosine_similarity(f_orig, f_recon).item())
    return np.mean(scores)

pixcorr = compute_pixcorr(recons, imgs_01)
ssim_val = compute_ssim(recons, imgs_01)
clip_sim = compute_clip_similarity(recons, imgs_01, clip_model, clip_proc, DEVICE)

print(f'PixCorr : {pixcorr:.4f}')
print(f'SSIM : {ssim_val:.4f}')
print(f'CLIP similarity: {clip_sim:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))

metric_names = ['PixCorr', 'SSIM', 'CLIP sim', '2-way acc']
metric_vals = [pixcorr, ssim_val, clip_sim, accs[0]]

bars = ax.bar(metric_names, metric_vals, color=['red', 'darkblue', 'green', 'orange'], width=0.7)
for bar, val in zip(bars, metric_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.03, f'{val:.3f}', ha='center', fontsize=11)

ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title(f'Evaluation metrics : {model_name}')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
metrics_path = os.path.join(results_dir, 'other_metrics', f'{decoder_name[:-3]}_metrics.png')
fig.savefig(metrics_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved : {metrics_path}')